# M3 승인 실험 — 실행 보류 상태

계정: **daulkim2001@gmail.com**. 2026-10-12 사용자의 최신 지시: GPU 실행 보류, 코드와 로컬 검증만 준비.

이 노트북은 기본 Run All로 GPU 학습·다운로드·데이터 업로드를 시작하지 않습니다. 향후 GPU 실행을 다시 승인한 뒤 런타임을 선택하고 RUN_GPU=True로 바꾸세요. 추가 단위 구매는 하지 않습니다. 여기에는 소스만 포함되며 비공개 전사·라벨·가중치는 없습니다.

목표: ① microbatch/global soft-F1 대조 ② 사람 근거 감독 ③ 누수 없는 OOF 증류 ④ 선택적 512토큰 재검토. 공식 Validation을 읽거나 제출본을 교체하는 단계는 없습니다.

In [ ]:
RUN_GPU = False
RUN_PREFLIGHT = False
PHASE = 'loss'  # loss / teachers / student / annotations / evidence / cascade
SEED = 42
MAX_HOURS = 3
HUMAN_ANNOTATION_RULES_CONFIRMED = False
ANNOTATIONS = None
print('GPU 실행:', RUN_GPU, '| 데이터 점검:', RUN_PREFLIGHT, '| 단계:', PHASE)


In [ ]:
import base64,hashlib,io,json,os,sys,zipfile,tarfile,subprocess
from pathlib import Path
ROOT=Path('/content/m3_approved')
ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_B64='...'
blob=base64.b64decode(SOURCE_B64)
assert hashlib.sha256(blob).hexdigest()=='fcd0ca0f7280c7e659854c11a2ade3d0b3e73ffc71112786da99f2dc915cc5fb'
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for name in archive.namelist():
        target=(ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
for name,digest in json.loads((ROOT/'source_manifest.json').read_text()).items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==digest
print('Source-only package verified:',len(json.loads((ROOT/'source_manifest.json').read_text())))


In [ ]:
if RUN_GPU or RUN_PREFLIGHT:
    os.environ['USE_TF']='0'
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','scikit-learn','iterative-stratification','numpy','pytest'],check=True)
    subprocess.run([sys.executable,'-m','pytest','tests/test_mission3_approved.py','-q'],cwd=ROOT,check=True)
else:
    print('패키지 설치와 테스트 실행 보류')


## 입력 준비

향후 실행할 때 기존 `input.tar.gz`를 Colab 세션 저장소에 업로드합니다. 내용은 Training 전사·9개 라벨, 기존 체크포인트, 내부 검증 확률입니다. 외부 API에 전사를 보내지 않습니다. 아래 업로드 셀은 RUN_GPU 또는 RUN_PREFLIGHT를 명시적으로 켠 경우에만 작동합니다.

In [ ]:
if RUN_GPU or RUN_PREFLIGHT:
    INPUT=Path('/content/input.tar.gz')
    if not INPUT.exists():
        from google.colab import files
        files.upload()
    assert INPUT.exists(),'input.tar.gz가 필요합니다'
    bundle=ROOT/'input'
    bundle.mkdir(exist_ok=True)
    if not (bundle/'bundle.json').exists():
        with tarfile.open(INPUT) as archive:
            members=[m for m in archive.getmembers() if m.name in ('baseline.pt','bundle.json') or m.name.startswith(('training/','reference/'))]
            archive.extractall(bundle,members=members,filter='data')
    command=[sys.executable,'scripts/train_mission3_approved.py','preflight','--bundle',str(bundle),'--out',str(ROOT/'run')]
    subprocess.run(command,cwd=ROOT,check=True)
else:
    print('비공개 데이터 업로드·읽기 보류')


## 실행 단계

- **loss**: 16·64 유효 배치 각각 micro8 vs global, 동일 체크포인트 1 epoch 연속 학습.
- **teachers → student**: Training 23,408건 안에서 새로운 3-fold 교사 학습. 29,200건 전체 OOF를 재사용하지 않습니다. 공개 사전학습 모델에서 시작하며 교사당 3 epochs. 학생 hard-label 대조군과 증류를 비교합니다. 비용이 큰 단계입니다.
- **annotations**: GPU 없이 Training 300통화의 검토 양식만 생성. 자동 규칙·AI 결과를 사람 주석으로 취급하지 않습니다.
- **evidence**: 사람이 검토한 JSONL과 추가 주석 규정 확인이 있어야 실행됩니다. 동일 구조 무감독 대조군 포함.
- **cascade**: 256·512 각각 동일 1 epoch 연속 학습. 내부 검증에서 5-fold 교차 적합으로 재검토 기준 선택.

단계별 실행 후 다음 PHASE를 선택합니다. 3시간은 각 단계가 아니라 동일 run 폴더 전체 누적 한도입니다. seed42 pilot이 유망하면 SEED=43/44로 반복하세요. 자동으로 채택하지 않습니다.

In [ ]:
if RUN_GPU:
    import torch
    assert torch.cuda.is_available(),'향후 승인 후 Colab GPU 런타임을 선택하세요'
    command=[sys.executable,'-u','scripts/train_mission3_approved.py',PHASE,'--bundle',str(ROOT/'input'),'--out',str(ROOT/'run'),'--seed',str(SEED),'--max-hours',str(MAX_HOURS),'--execute']
    if PHASE=='evidence':
        assert HUMAN_ANNOTATION_RULES_CONFIRMED and ANNOTATIONS,'사람 주석 및 규정 확인이 필요합니다'
        command+=['--annotations',str(ANNOTATIONS),'--human-annotation-rules-confirmed']
    log=ROOT/'run'/'execution.log'
    log.parent.mkdir(exist_ok=True)
    with log.open('a',encoding='utf-8') as stream:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            print(line,end='',flush=True)
            stream.write(line)
            stream.flush()
        assert process.wait()==0,'단계 중단: execution.log와 budget.json 확인'
elif RUN_PREFLIGHT and PHASE=='annotations':
    subprocess.run([sys.executable,'scripts/train_mission3_approved.py','annotations','--bundle',str(ROOT/'input'),'--out',str(ROOT/'run')],cwd=ROOT,check=True)
else:
    print('학습을 시작하지 않았습니다.')


In [ ]:
if (ROOT/'run').exists():
    subprocess.run([sys.executable,'scripts/summarize_mission3_approved.py','--run',str(ROOT/'run')],cwd=ROOT,check=True)
else:
    print('아직 실제 학습 결과 없음')


## 보존과 CPU 속도 측정

학습 뒤 모델과 결과를 모두 확보하기 전에는 런타임을 삭제하지 않습니다. 다음 셀은 별도로 DOWNLOAD_RESULTS=True를 지정해야 내려받습니다. resume.pt는 optimizer를 포함하므로 크며, 다음 세션에서 학습을 재개할 때는 별도로 보존해야 합니다.

로컬 속도 측정은 `scripts/inference_mission3_approved.py --ckpt-path ... --label-dir TrainingSubset --output predictions.csv --limit 200 --threads 4`로 합니다. 교사 전체는 --ckpt-path 대신 --teacher-dir run/teachers로 측정합니다. 같은 PC·동일 표본·동일 스레드로 반복 측정하고 total_seconds_including_load를 비교합니다. 단일 모델 warm-up과 교사 해시 검사 포함 여부가 달라 장기 상주 추론 성능으로 일반화하지 않습니다.

In [ ]:
DOWNLOAD_RESULTS=False
if DOWNLOAD_RESULTS:
    from google.colab import files
    result=ROOT/'run'
    assert result.exists()
    manifest={}
    for path in result.rglob('model.pt'):
        digest=hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda:stream.read(1<<20),b''):digest.update(block)
        manifest[str(path.relative_to(result))]={'bytes':path.stat().st_size,'sha256':digest.hexdigest()}
    (result/'checkpoint_manifest.json').write_text(json.dumps(manifest,indent=2))
    archive_path=ROOT/'results.zip'
    with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in result.rglob('*'):
            if path.is_file() and path.suffix in ('.json','.npz','.log','.md','.joblib'):
                archive.write(path,path.relative_to(result))
    files.download(str(archive_path))
    for name in manifest: files.download(str(result/name))
    print('체크포인트 다운로드 후 로컬에서 크기와 SHA256을 대조하세요. GPU 세션은 자동 삭제하지 않습니다.')
